## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install lightgbm xgboost -q

Mounted at /content/drive


In [2]:
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                             median_absolute_error, mean_absolute_percentage_error,
                             explained_variance_score)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RAW  = "/content/drive/MyDrive/Machine_learning_project/Data/Raw"
PROC = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"
SRC  = "/content/drive/MyDrive/Machine_learning_project/src"

## Load the feature bundle

In [3]:
b = np.load(f"{PROC}/feature_bundle.npz", allow_pickle=True)

X_tab, X_text, X_img = b["X_tab"], b["X_text"], b["X_img"]
all_ids   = b["all_ids"]
train_ids, test_ids = b["train_ids"], b["test_ids"]
y_train,   y_test   = b["y_train"],  b["y_test"]

row_of = {i: k for k, i in enumerate(all_ids)}
rows   = lambda ids: [row_of[i] for i in ids]
tr_rows, te_rows = rows(train_ids), rows(test_ids)

print("tabular:", X_tab.shape, "| text:", X_text.shape, "| image:", X_img.shape)

tabular: (6158, 152) | text: (6158, 100) | image: (6158, 50)


## The 7 feature sets and the 7 models

In [4]:
FEATURES = {
    "tabular":            X_tab,
    "text":               X_text,
    "image":              X_img,
    "tabular+text":       np.hstack([X_tab, X_text]),
    "tabular+image":      np.hstack([X_tab, X_img]),
    "text+image":         np.hstack([X_text, X_img]),
    "tabular+text+image": np.hstack([X_tab, X_text, X_img]),
}

def make_models():
    return {
        "Dummy":        DummyRegressor(strategy="mean"),
        "Ridge":        Ridge(alpha=1.0),
        "kNN":          KNeighborsRegressor(n_neighbors=15),
        "RandomForest": RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42),
        "XGBoost":      XGBRegressor(n_estimators=800, max_depth=6, learning_rate=0.05,
                                     subsample=0.8, colsample_bytree=0.8,
                                     tree_method="hist", n_jobs=-1, random_state=42),
        "LightGBM":     LGBMRegressor(n_estimators=800, max_depth=6, learning_rate=0.05,
                                      subsample=0.8, colsample_bytree=0.8,
                                      n_jobs=-1, random_state=42, verbose=-1),
        "MLP":          "torch",     # handled separately (see next cell)
    }

# these are NOT scale-invariant, so they get standardised features
SCALE_SENSITIVE = {"Ridge", "kNN", "MLP"}

for k, v in FEATURES.items():
    print(f"{k:<20} {v.shape}")

tabular              (6158, 152)
text                 (6158, 100)
image                (6158, 50)
tabular+text         (6158, 252)
tabular+image        (6158, 202)
text+image           (6158, 150)
tabular+text+image   (6158, 302)


## The MLP (PyTorch)

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

def train_mlp(Xtr_np, ytr_np, Xte_np, seed=42, epochs=250):
    """3-layer MLP. Returns test predictions."""
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(seed)

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32, device=dev)
    Xte = torch.tensor(Xte_np, dtype=torch.float32, device=dev)
    ytr = torch.tensor(ytr_np, dtype=torch.float32, device=dev).view(-1, 1)

    layers, d = [], Xtr.shape[1]
    for h in (256, 128, 64):
        layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)]
        d = h
    layers.append(nn.Linear(d, 1))
    net = nn.Sequential(*layers).to(dev)

    opt  = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.MSELoss()
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=128, shuffle=True)

    net.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()             # gradients accumulate by default!
            crit(net(xb), yb).backward()
            opt.step()

    net.eval()
    with torch.no_grad():
        return net(Xte).cpu().numpy().ravel()

print("device:", "cuda" if torch.cuda.is_available() else "cpu")

device: cuda


## Metrics

Every metric on **both scales**:

- **log scale** — what the model actually optimises; use this to compare models.
- **dollar scale** — what a human understands; use this to describe the error.

| Metric | Meaning |
|---|---|
| R2 | fraction of variance explained (1.0 = perfect, 0.0 = no better than the mean) |
| ExplVar | like R2 but ignores systematic bias |
| RMSE | root mean squared error — punishes big misses hard |
| MAE | mean absolute error — the *typical* miss |
| MedAE | median absolute error — robust to a few terrible predictions |
| MAPE | mean absolute percentage error |

In [6]:
def all_metrics(y_true_log, y_pred_log):
    yt, yp = np.exp(y_true_log), np.exp(y_pred_log)     # back to dollars
    return {
        "R2":        r2_score(y_true_log, y_pred_log),
        "ExplVar":   explained_variance_score(y_true_log, y_pred_log),
        "RMSE_log":  np.sqrt(mean_squared_error(y_true_log, y_pred_log)),
        "MAE_log":   mean_absolute_error(y_true_log, y_pred_log),
        "MedAE_log": median_absolute_error(y_true_log, y_pred_log),
        "RMSE_usd":  np.sqrt(mean_squared_error(yt, yp)),
        "MAE_usd":   mean_absolute_error(yt, yp),
        "MedAE_usd": median_absolute_error(yt, yp),
        "MAPE":      mean_absolute_percentage_error(yt, yp),
    }

## Run the grid



In [7]:
results, predictions = [], {}

for feat_name, X in FEATURES.items():
    # scaled copy for the scale-sensitive models — FIT ON TRAIN ONLY
    sc = StandardScaler().fit(X[tr_rows])
    Xs = sc.transform(X)

    for model_name, model in make_models().items():
        use = Xs if model_name in SCALE_SENSITIVE else X
        Xtr, Xte = use[tr_rows], use[te_rows]

        t0 = time.time()
        if model_name == "MLP":
            pred = train_mlp(Xtr, y_train, Xte)
        else:
            model.fit(Xtr, y_train)
            pred = model.predict(Xte)
        elapsed = time.time() - t0

        m = all_metrics(y_test, pred)
        results.append({
            "features": feat_name,
            "model": model_name,
            "n_features": X.shape[1],
            "fit_seconds": round(elapsed, 1),
            **{k: round(v, 4) for k, v in m.items()},
        })
        predictions[f"{feat_name}|{model_name}"] = pred
        print(f"{feat_name:<20} {model_name:<14} R2={m['R2']:6.3f}  ({elapsed:5.1f}s)")

results = pd.DataFrame(results).sort_values("R2", ascending=False)
print("\ndone —", len(results), "runs")

tabular              Dummy          R2=-0.004  (  0.0s)
tabular              Ridge          R2= 0.672  (  0.2s)
tabular              kNN            R2= 0.462  (  0.4s)
tabular              RandomForest   R2= 0.714  ( 56.0s)
tabular              XGBoost        R2= 0.742  (  5.5s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


tabular              LightGBM       R2= 0.738  (  1.0s)
tabular              MLP            R2= 0.716  ( 40.5s)
text                 Dummy          R2=-0.004  (  0.0s)
text                 Ridge          R2= 0.547  (  0.0s)
text                 kNN            R2= 0.450  (  0.1s)
text                 RandomForest   R2= 0.530  (260.9s)
text                 XGBoost        R2= 0.578  ( 28.0s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


text                 LightGBM       R2= 0.590  (  7.8s)
text                 MLP            R2= 0.608  ( 34.0s)
image                Dummy          R2=-0.004  (  0.0s)
image                Ridge          R2= 0.180  (  0.0s)
image                kNN            R2= 0.129  (  0.0s)
image                RandomForest   R2= 0.250  (131.5s)
image                XGBoost        R2= 0.250  ( 13.1s)


...

tabular+image        LightGBM       R2= 0.732  (  4.8s)
tabular+image        MLP            R2= 0.711  ( 32.6s)
text+image           Dummy          R2=-0.004  (  0.0s)
text+image           Ridge          R2= 0.551  (  0.0s)
text+image           kNN            R2= 0.401  (  0.1s)
text+image           RandomForest   R2= 0.516  (369.3s)
text+image           XGBoost        R2= 0.592  ( 42.3s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


text+image           LightGBM       R2= 0.591  (  7.6s)
text+image           MLP            R2= 0.572  ( 34.4s)
tabular+text+image   Dummy          R2=-0.004  (  0.0s)
tabular+text+image   Ridge          R2= 0.706  (  0.0s)
tabular+text+image   kNN            R2= 0.431  (  0.2s)
tabular+text+image   RandomForest   R2= 0.704  (433.0s)
tabular+text+image   XGBoost        R2= 0.750  ( 43.7s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


tabular+text+image   LightGBM       R2= 0.751  ( 10.7s)
tabular+text+image   MLP            R2= 0.713  ( 33.5s)

done — 49 runs


## Stacking model

In [8]:
# ============================================================
# 6b. STACKING — prediction-level fusion (model #5)
#
# The base models' TRAIN predictions must be out-of-fold: each prediction
# comes from a model that never saw that row. Otherwise the meta-model
# learns to trust predictions that are unrealistically good, and the
# result is leaked and meaningless.
# ============================================================
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.linear_model import RidgeCV
from sklearn.base import clone

def xgb():
    return XGBRegressor(n_estimators=800, max_depth=6, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8,
                        tree_method="hist", n_jobs=-1, random_state=42)

BASE = {"tabular": X_tab, "text": X_text, "image": X_img}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

meta_train, meta_test = [], []

for name, X in BASE.items():
    Xtr, Xte = X[tr_rows], X[te_rows]

    # TRAIN meta-features: out-of-fold predictions
    oof = cross_val_predict(xgb(), Xtr, y_train, cv=kf)
    meta_train.append(oof)

    # TEST meta-features: fit on ALL train, predict the untouched test set
    m = xgb().fit(Xtr, y_train)
    meta_test.append(m.predict(Xte))

    print(f"{name:<10} OOF R2: {r2_score(y_train, oof):6.3f} | "
          f"test R2: {r2_score(y_test, meta_test[-1]):6.3f}")

meta_train = np.column_stack(meta_train)   # (4926, 3)
meta_test  = np.column_stack(meta_test)    # (1232, 3)
print("meta-features:", meta_train.shape, "->", meta_test.shape)

tabular    OOF R2:  0.776 | test R2:  0.742
text       OOF R2:  0.579 | test R2:  0.578
image      OOF R2:  0.240 | test R2:  0.250
meta-features: (4926, 3) -> (1232, 3)


In [9]:
# the meta-model learns how much to trust each modality's prediction
meta = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0]).fit(meta_train, y_train)
pred_stack = meta.predict(meta_test)

m = all_metrics(y_test, pred_stack)
print(f"STACKING   R2: {m['R2']:.3f} | log-RMSE: {m['RMSE_log']:.3f}\n")

# what did it learn to trust? -> a direct, quantitative read on modality value
for name, w in zip(BASE.keys(), meta.coef_):
    print(f"  weight[{name:<8}]: {w:+.3f}")

# --- append to the results table so it lands in model_results.csv ---
results = pd.concat([
    results,
    pd.DataFrame([{
        "features":    "STACKING (tab+text+img)",
        "model":       "Ridge-meta",
        "n_features":  3,
        "fit_seconds": 0.0,
        **{k: round(v, 4) for k, v in m.items()},
    }])
], ignore_index=True).sort_values("R2", ascending=False)

predictions["STACKING (tab+text+img)|Ridge-meta"] = pred_stack
print("\nappended to results — now", len(results), "rows")

STACKING   R2: 0.753 | log-RMSE: 0.287

  weight[tabular ]: +0.857
  weight[text    ]: +0.177
  weight[image   ]: +0.083

appended to results — now 50 rows


## Save results

In [10]:
results.to_csv(f"{PROC}/model_results.csv", index=False)
np.savez_compressed(f"{PROC}/predictions.npz", y_test=y_test, **predictions)

print("saved model_results.csv and predictions.npz")
results.head(12)

saved model_results.csv and predictions.npz


,features,model,n_features,fit_seconds,R2,ExplVar,RMSE_log,MAE_log,MedAE_log,RMSE_usd,MAE_usd,MedAE_usd,MAPE
0,tabular+text,LightGBM,252,6.0,0.7530,0.7545,0.2867,0.2046,0.1518,91.1884,42.2735,21.5392,0.1981
49,STACKING (tab+text+img),Ridge-meta,3,0.0,0.7529,0.7542,0.2867,0.2019,0.1516,91.2130,41.8523,21.0905,0.1959
1,tabular+text,XGBoost,252,29.6,0.7520,0.7535,0.2872,0.2042,0.1483,90.6715,41.8821,21.2489,0.1978
2,tabular+text+image,LightGBM,302,10.7,0.7514,0.7531,0.2876,0.2049,0.1521,90.7691,42.1275,21.4109,0.1986
3,tabular+text+image,XGBoost,302,43.7,0.7496,0.7506,0.2887,0.2053,0.1493,91.0745,42.3821,21.0710,0.2006
4,tabular,XGBoost,152,5.5,0.7420,0.7433,0.2930,0.2070,0.1523,91.2818,42.6585,21.7364,0.2014
5,tabular,LightGBM,152,1.0,0.7381,0.7396,0.2952,0.2126,0.1608,90.5088,43.6262,22.5357,0.2069
6,tabular+image,XGBoost,202,16.7,0.7348,0.7361,0.2971,0.2148,0.1574,91.8234,44.2585,21.9677,0.2106
7,tabular+image,LightGBM,202,4.8,0.7321,0.7335,0.2986,0.2141,0.1588,91.4773,44.0367,23.2127,0.2093
8,tabular+text,MLP,252,34.1,0.7307,0.7376,0.2994,0.2144,0.1642,90.1647,43.6549,21.5611,0.2058


## Quick look — the headline table

In [11]:
pivot = results.pivot(index="features", columns="model", values="R2")
order = ["tabular", "text", "image", "tabular+text", "tabular+image",
         "text+image", "tabular+text+image"]
pivot.reindex(order)

model,Dummy,LightGBM,MLP,RandomForest,Ridge,Ridge-meta,XGBoost,kNN
features,,,,,,,,
tabular,-0.0038,0.7381,0.7161,0.7145,0.6718,NaN,0.7420,0.4617
text,-0.0038,0.5901,0.6080,0.5299,0.5467,NaN,0.5780,0.4498
image,-0.0038,0.2310,0.1804,0.2496,0.1804,NaN,0.2502,0.1289
tabular+text,-0.0038,0.7530,0.7307,0.7077,0.7048,NaN,0.7520,0.4464
tabular+image,-0.0038,0.7321,0.7114,0.7018,0.6776,NaN,0.7348,0.4086
text+image,-0.0038,0.5914,0.5718,0.5159,0.5511,NaN,0.5920,0.4010
tabular+text+image,-0.0038,0.7514,0.7127,0.7044,0.7058,NaN,0.7496,0.4307
